# Aspire : garde-fous du code d'agent — l'analyseur Roslyn vit DANS la compilation

Nos notebooks précédents (01-05) ont construit une pile d'agent .NET complète : orchestration Aspire, stack GenAI réelle, observabilité, agent streaming, tests d'intégration. Ce notebook attaque la question qui vient après : **quand un agent (LLM) génère du code C# pour cette pile, qu'est-ce qui l'empêche d'écrire du code dangereux ?**

La thèse tient en une ligne de parité :

| Python | C#/.NET |
|---|---|
| `mypy`, `ruff` — garde-fous **HORS** compilation (outil séparé à adopter) | **analyseurs Roslyn** — garde-fous **DANS** la compilation (`dotnet build` lui-même rend le diagnostic) |

En Python, un garde-fou vit dans un lint distinct : il faut installer `ruff`, le configurer, l'ajouter au pipeline. En .NET, l'analyseur se référence comme n'importe quelle dépendance — et le **même `dotnet build` qui produit le binaire** rend le verdict, pour tout le monde, sur la machine de chaque développeur et du premier coup.

**Plan** : nous construisons un vrai analyseur (`DiagnosticAnalyzer` Roslyn, ~40 lignes) qui attrape le pattern de deadlock le plus typique du code d'agent généré — bloquer une `Task` avec `.Result`/`.Wait()` — puis nous le voyons tirer sur deux canaux : le canal *build* (`dotnet build` rend l'avertissement, c'est la thèse) et le canal *API* (un `Verifier` qui compile un source via Roslyn et rend son verdict, comme le fait l'IDE). Nous terminons par le contraste Python et trois exercices.

## Contexte : le deadlock `.Result`/`.Wait()`, blessure classique du code d'agent

Un LLM qui génère du C# « dans le style synchrone » écrit très naturellement :

```csharp
var reponse = AppelerLeLlmAsync(prompt).Result;   // bloque la Task
```

Pourquoi c'est mortel et pas juste inélégant : `.Result` **bloque le thread courant** jusqu'à ce que la tâche finisse. Si cette tâche a besoin, pour finir, d'un thread du même pool — continuation en attente, contexte de synchronisation, handler ASP.NET qui doit libérer son thread — le programme **attend un thread qui ne viendra jamais** : deadlock. Symptôme en production : le service se fige silencieusement, aucun crash, aucun log — juste zéro réponse.

C'est précisément le genre de défaut qu'un garde-fou automatique doit attraper : *localement visible* (une expression), *globalement désastreux* (un service mort). Et c'est un pattern que les modèles de langage produisent de façon récurrente, parce que leurs données d'entraînement débordent de code synchrone. D'où la règle AGENTGUARD001 : **toute Task bloquée synchrone en code d'agent est signalée au build**.

In [1]:
using System;
using System.Diagnostics;
using System.IO;

var here = Directory.GetCurrentDirectory();   // le dossier de la serie Aspire

public static class Shell
{
    // Execute un executable et capture stdout+stderr. Pas de shell
    // intermediaire : cmd.exe reecrit toute ligne portant plus d'une paire
    // de guillemets (lecon du notebook 01) -- on lance donc la commande
    // directement, resolue via le PATH.
    public static string Run(string workDir, string cmd, string args)
    {
        var psi = new ProcessStartInfo
        {
            FileName = cmd,
            Arguments = args,
            WorkingDirectory = workDir,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            UseShellExecute = false,
            CreateNoWindow = true
        };
        using var p = Process.Start(psi)!;
        var stdout = p.StandardOutput.ReadToEnd();
        var stderr = p.StandardError.ReadToEnd();
        if (!p.WaitForExit(180_000)) p.Kill();
        return stdout + stderr;
    }
}
Console.WriteLine($"Repertoire de travail : {Path.GetFileName(here)}");

Repertoire de travail : Aspire


## A1 — Le terrain fautif : un worker d'agent qui bloque sa tâche

Le fichier ci-dessous (`AgentGuard.Demo/Program.cs`, committé avec ce notebook) est une **copie de démonstration** du motif canonique de la série (le worker `Channel` du notebook 04) — avec le défaut **injecté** : là où la série attend la tâche avec `await`, ce code la bloque avec `.Result`, deux fois, « pour simplifier ». C'est le genre de code qu'un agent génère quand on lui demande une version synchrone d'un pipeline asynchrone.

Le fichier ne fait pas planter le notebook : l'exécution du programme elle-même termine (blocage borné sur ce scénario minimal) — le défaut est un **risque structurel**, pas une exception immédiate. C'est exactement pourquoi il faut un analyseur : aucun test d'exécution ne le révélera tant que le deadlock ne s'est pas produit.

In [2]:
var terrain = File.ReadAllText(Path.Combine(here, "AgentGuard.Demo", "Program.cs"));
Console.WriteLine(terrain);

// Terrain fautif : copie de demo du motif StreamingAgent.App (notebook 04).
// Un worker d'agent consomme une Channel -- motif canonique de la serie --
// mais ici le code genere "dans le style synchrone" BLOQUE la tache d'appel
// au LLM avec .Result au lieu de l'attendre. C'est le defaut que l'analyseur
// AGENTGUARD001 doit attraper au `dotnet build` (ce fichier ne leve PAS
// d'exception : le blocage ne deadlock ici qu'un contexte reduit -- le
// notebook explique pourquoi le pattern reste mortel en production).

using System;
using System.Threading.Channels;
using System.Threading.Tasks;

var channel = Channel.CreateUnbounded<string>();
_ = Producer.ProduceAsync(channel.Writer, "traduis cette phrase en anglais");

Console.WriteLine(AgentWorker.TranslateSync(channel.Reader));

public static class AgentWorker
{
    // Genere par agent "pour simplifier" : la tache async est bloquee.
    // Deux blocages .Result -- deux diagnostics attendus au build.
    public static string Translat

### Lecture du terrain

Le worker `TranslateSync` porte les deux blocages, marqués `// AGENTGUARD001` :

- `inbound.ReadAsync().AsTask().Result` — on bloque même la **lecture du canal** d'entrée ;
- `CallLlmAsync(prompt).Result` — on bloque **l'appel au LLM**, la partie la plus lente du pipeline.

Deux expressions, deux endroits où un thread s'endort en tenant le pipeline. La méthode `CallLlmAsync` est correcte (elle est `async`) ; c'est le **consommateur** qui dénature le contrat. Un humain relit rarement ça assez attentivement — l'analyseur, lui, ne rate jamais.

## A2 — L'analyseur : quarante lignes qui vivent dans la compilation

Voici l'intégralité de l'analyseur (`AgentGuard.Analyzers/TaskResultBlockAnalyzer.cs`, committé) — un `DiagnosticAnalyzer` Roslyn réel, pas une simplification :

Il s'abonne aux **expressions d'accès de membre** (`.QuelqueChose`), puis filtre en deux étages : un étage **syntaxique** bon marché (le membre s'appelle-t-il `Result` ou `Wait` ?) et un étage **sémantique** (ce membre appartient-il vraiment au type `Task`/`Task<T>` ?). C'est ce second étage qui fait la différence entre un `grep` et un analyseur : il interroge le **modèle sémantique** — la compréhension des types que le compilateur construit.

In [3]:
var analyzerSrc = File.ReadAllText(Path.Combine(here, "AgentGuard.Analyzers", "TaskResultBlockAnalyzer.cs"));
Console.WriteLine(analyzerSrc);

using System.Collections.Immutable;
using Microsoft.CodeAnalysis;
using Microsoft.CodeAnalysis.CSharp;
using Microsoft.CodeAnalysis.Diagnostics;

namespace AgentGuard.Analyzers;

/// <summary>
/// AGENTGUARD001 : blocage synchrone d'une Task (.Result / .Wait()).
///
/// Pattern typique du code d'agent genere : appeler une tache asynchrone
/// (appel LLM, streaming, canal) depuis du code synchrone en la bloquant.
/// Le garde-fou vit DANS la compilation -- `dotnet build` rend le diagnostic
/// sans aucun outil supplementaire (la these du notebook 06 de la serie Aspire,
/// Epic #10473 axe Roslyn).
/// </summary>
[DiagnosticAnalyzer(LanguageNames.CSharp)]
public sealed class TaskResultBlockAnalyzer : DiagnosticAnalyzer
{
    public const string DiagnosticId = "AGENTGUARD001";

    private static readonly DiagnosticDescriptor Rule = new(
        DiagnosticId,
        "Blocage synchrone d'une Task",
        "Task bloquee de maniere synchrone ({0}) : deadlock potentiel en code d'agent",
   

### Anatomie pas à pas

Trois morceaux portent tout :

1. **`DiagnosticDescriptor Rule`** — la carte d'identité du diagnostic : identifiant (`AGENTGUARD001`), titre, message (avec `{0}` rempli au rapport), catégorie, sévérité. `isEnabledByDefault: true` : le diagnostic est actif dès la référence, sans configuration.
2. **`Initialize` + `RegisterSyntaxNodeAction`** — l'analyseur ne parcourt PAS tout le code : il s'abonne à un seul type de nœud syntaxique (`SimpleMemberAccessExpression`), et Roslyn ne l'appelle que sur ces nœuds. C'est ce qui rend l'analyse quasi gratuite.
3. **`AnalyzeNode`** — les deux étages de filtrage : syntaxique d'abord (`Result` ou `Wait` — un test de chaîne), sémantique ensuite (`GetSymbolInfo` → le membre appartient-il à `Task`/`Task<T>` ?). Le `MetadataName` distingue le vrai `Task<T>` (`` `Task`1` `` en métadonnées) de tout homonyme. Seul un nœud ayant passé les deux étages est rapporté — d'où zéro faux positif sur un type custom qui aurait une propriété `Result`.

La sévérité `Warning` (et non `Error`) est un choix : le build ne casse pas, mais le diagnostic est visible à chaque compilation. Passer en `Error` (via `.editorconfig`) ferait du garde-fou un bloqueur — une décision d'équipe, pas de l'analyseur.

### Exercice 1 -- Predire le verdict du Verifier sur le terrain a exemptions

L'analyseur AGENTGUARD005b est livre, ses exemptions sont posees explicitement. Le terrain `SyncOverAsyncConfigureAwaitValueTask.cs` materialise **trois cas d'exemption** -- tous les trois legitimes. **TODO etudiant** : avant d'executer la cellule Verifier, predire le verdict de chacun (rouge / propre), puis executer, puis **citer pour chaque cas le numero de la clause d'exemption dans `SyncOverAsyncConfigureAwaitAnalyzer.cs`** (le filtre syntaxique, le filtre semantique, ou l'absence de `GetResult`). Relier la clause citee au type de defense (anti-faux-positif).

In [2]:
// Exercice 1 -- prediction AVANT execution.
//
// Le terrain `SyncOverAsyncConfigureAwaitValueTask.cs` porte trois
// cas. Pour CHACUN, predire le verdict AGENTGUARD005b (rouge / propre)
// PUIS executer le Verifier sur le terrain et confronter :
//
//   var predits = new (string cas, string verdictAttendu, string clauseExemption)[]
//   {
//       ("ValueTaskSync", "PROPRE", "filtre semantique : ValueTask n'est pas Task"),
//       ("AvecCondition", "PROPRE", "filtre syntaxique : argument n'est pas un literal bool"),
//       ("SansGetResult", "PROPRE", "filtre syntaxique : pas de GetResult dans la chaine"),
//   };
//   foreach (var p in predits) Console.WriteLine($"{p.cas,-15} attendu={p.verdictAttendu,-7} | {p.clauseExemption}");
//
// L'analyseur distingue 3 systemes de defense orthogonaux. Une seule
// clause suffit a blanchir un cas -- ce qui rend les faux-positifs
// rares, mais pas gratuits : les nommer dans la reponse est la
// moitie de l'exercice.

var verdictsValueTask = Shell.Run(here, "dotnet",
    "run --project AgentGuard.Verifier -- " +
    "AgentGuard.Verifier/samples/SyncOverAsyncConfigureAwaitValueTask.cs");
Console.WriteLine(verdictsValueTask);

Console.WriteLine("Exercice a completer : citer pour chaque cas la clause d'exemption gagnee");


usage: dotnet run -- <fautif.cs> <corrige.cs> [autres.cs ...]



Exercice a completer : citer pour chaque cas la clause d'exemption gagnee


## A3 — La thèse : `dotnet build` rend le diagnostic, sans aucun outil supplémentaire

Le projet `AgentGuard.Demo` référence l'analyseur **comme dépendance de diagnostic** — deux attributs dans le `.csproj` :

```xml
<ProjectReference Include="../AgentGuard.Analyzers/AgentGuard.Analyzers.csproj"
                  OutputItemType="Analyzer" ReferenceOutputAssembly="false" />
```

`OutputItemType="Analyzer"` dit à MSBuild : « charge cette DLL comme analyseur Roslyn de CE projet ». `ReferenceOutputAssembly="false"` ajoute : « mais elle n'est pas une dépendance runtime ». C'est tout. Pas de paquet à installer, pas de ligne de commande à retenir, aucune action consciente à faire — le prochain `dotnet build` rend le verdict. C'est la thèse en action.

In [5]:
// -t:Rebuild : force la recompilation complete -- un build incrementale
// (projet deja compile par un passage precedent) ne re-emet PAS les
// diagnostics, et le verdict doit apparaitre a CHAQUE execution.
// MSBuild rend des chemins absolus (machine-dependants) dans chaque
// diagnostic : on les abrege en chemin relatif au dossier de la serie
// (le suffixe [projet.csproj] garde son nom court) -- motif #11859.
var buildRaw = Shell.Run(here, "dotnet", "build AgentGuard.Demo -t:Rebuild -v q --nologo");
var build = string.Join(Environment.NewLine, buildRaw.Split('\n').Select(line =>
{
    var l = line.TrimEnd('\r');
    l = l.Replace(here + Path.DirectorySeparatorChar, "");
    // Suffixe [chemin\projet.csproj] -> [nom du projet]
    var open = l.LastIndexOf('[');
    if (open >= 0 && l.EndsWith(".csproj]"))
        l = l[..open] + $"[{Path.GetFileName(l[(open + 1)..^8])}]";
    return l;
}));
Console.WriteLine(build);

AgentGuard.Demo\Program.cs(24,22): warning AGENTGUARD001: Task bloquee de maniere synchrone (.Result) : deadlock potentiel en code d'agent [AgentGuard.Demo]
AgentGuard.Demo\Program.cs(25,16): warning AGENTGUARD001: Task bloquee de maniere synchrone (.Result) : deadlock potentiel en code d'agent [AgentGuard.Demo]
AgentGuard.Demo\Program.cs(88,15): warning AGENTGUARD004: L'appel à 'Task<string> AgentCancellation.CallLlmAsync(string prompt, CancellationToken cancellationToken = default(CancellationToken))' omet CancellationToken alors que 'cancellationToken' est disponible dans la méthode englobante [AgentGuard.Demo]
AgentGuard.Demo\Program.cs(55,30): warning AGENTGUARD002: La methode async void 'SurveillerCanalAsync' echappe a toute attente -- exceptions non observees, process mort [AgentGuard.Demo]
AgentGuard.Demo\Program.cs(75,9): warning AGENTGUARD003: Task.Run 'Task.Run(() => Console.WriteLine("ping"))' lance une tache non observee -- exceptions perdues, comportement indefini [AgentG

### Lecture du verdict

La sortie du build porte désormais **cinq avertissements**, un par défaut démontré :

- `AGENTGUARD001 x2` — les deux `.Result` qui bloquent la lecture du canal puis l'appel LLM ;
- `AGENTGUARD002 x1` — la méthode `async void SurveillerCanalAsync` hors gestionnaire d'événement ;
- `AGENTGUARD003 x1` — le `Task.Run(...)` lancé comme énoncé autonome ;
- `AGENTGUARD004 x1` — l'appel LLM omet le `CancellationToken` pourtant disponible et accepté par la signature cible.

Les quatre analyseurs tirent dans le même build, sans configuration supplémentaire : chaque garde-fou ajouté au projet `AgentGuard.Analyzers` est automatiquement actif. Chacun nomme le fichier, la ligne, la colonne et le message complet. La dernière ligne — `0 Erreur(s)` — rappelle le choix de sévérité : le build réussit, le diagnostic est un garde-fou, pas un marteau.

**Le point décisif** : cette sortie est produite par `dotnet build`, la commande la plus ordinaire du monde .NET. Aucune mention d'outil externe, aucune étape de lint. Le diagnostic est arrivé **avec la compilation elle-même** — c'est ce que Python ne peut pas offrir par construction, et que la section C détaille.

## B1 — Le canal API : le `Verifier` compile un source et rend son verdict

`dotnet build` est le canal *intégration*. Il en existe un second : l'**API Roslyn** elle-même — c'est ce qu'utilisent l'IDE (le soulignement jaune pendant la frappe) et les tests. Le projet `AgentGuard.Verifier` (committé) prend deux fichiers sources en argument, les compile en mémoire via `CSharpCompilation` + `WithAnalyzers`, et rend un verdict par fichier.

On lui soumet les deux variantes du terrain : le fichier fautif du Demo, et la version corrigée (`samples/AgentWorkerCorrige.cs`, committée — le même worker, mais `async`/`await` de bout en bout).

In [6]:
var verdicts = Shell.Run(here, "dotnet",
    "run --project AgentGuard.Verifier -- AgentGuard.Demo/Program.cs AgentGuard.Verifier/samples/AgentWorkerCorrige.cs");
Console.WriteLine(verdicts);

[Program.cs] VERDICT : 5 diagnostic(s) -- AGENTGUARD001 x2, AGENTGUARD002 x1, AGENTGUARD003 x1, AGENTGUARD004 x1
    AGENTGUARD001 @ 24:22  Task bloquee de maniere synchrone (.Result) : deadlock potentiel en code d'agent
    AGENTGUARD001 @ 25:16  Task bloquee de maniere synchrone (.Result) : deadlock potentiel en code d'agent
    AGENTGUARD002 @ 55:30  La methode async void 'SurveillerCanalAsync' echappe a toute attente -- exceptions non observees, process mort
    AGENTGUARD003 @ 75:9  Task.Run 'Task.Run(() => Console.WriteLine("ping"))' lance une tache non observee -- exceptions perdues, comportement indefini
    AGENTGUARD004 @ 88:15  L'appel à 'Task<string> AgentCancellation.CallLlmAsync(string prompt, CancellationToken cancellationToken = default(CancellationToken))' omet CancellationToken alors que 'cancellationToken' est disponible dans la méthode englobante
[AgentWorkerCorrige.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche



### Lecture des verdicts

Deux lignes de verdict, chacune adossée à son fichier :

- `Program.cs` (le terrain fautif) : **5 diagnostics** — `AGENTGUARD001 x2`, puis un diagnostic pour chacun des garde-fous `002`, `003` et `004` ;
- `AgentWorkerCorrige.cs` (la version corrigée) : **PROPRE**, aucun garde-fou déclenché — son `CancellationToken` voyage déjà de bout en bout.

**Deux canaux, un seul moteur** : les analyseurs sont identiques ; ce qui change, c'est qui les héberge — MSBuild pour le canal build, l'API pour le canal tests/IDE. Un test d'intégration continue peut faire exactement ce que fait ce Verifier : compiler les sources générés par l'agent et exiger zéro diagnostic AgentGuard. Le garde-fou devient alors **exécutable en CI sur du code qui n'existe même pas sous forme de projet** — génération, compilation en mémoire, verdict, puis conservation ou régénération.

In [7]:
var corrige = File.ReadAllText(Path.Combine(here, "AgentGuard.Verifier", "samples", "AgentWorkerCorrige.cs"));
Console.WriteLine(corrige);

// Version corrigee du terrain fautif (AgentGuard.Demo/Program.cs) :
// le worker attend la tache au lieu de la bloquer. C'est la version
// attendue "propre" au verdict du Verifier.

using System;
using System.Threading;
using System.Threading.Channels;
using System.Threading.Tasks;

public static class AgentWorkerCorrige
{
    // Le fix : async/await de bout en bout. Le thread rend la main pendant
    // l'attente au lieu de se bloquer -- plus de deadlock possible, et le
    // pipeline reste fluide sous charge.
    public static async Task<string> TranslateAsync(
        ChannelReader<string> inbound, CancellationToken ct = default)
    {
        var prompt = await inbound.ReadAsync(ct);
        return await CallLlmAsync(prompt, ct);
    }

    private static async Task<string> CallLlmAsync(string prompt, CancellationToken ct)
    {
        await Task.Delay(50, ct);       // simule la latence de l'appel LLM
        return $"[LLM] {prompt}";
    }
}



### Le fix : `await`, et pourquoi il suffit

La version corrigée ne change pas la logique — seulement le contrat :

- `TranslateSync` devient `TranslateAsync` : `async Task<string>` au lieu de `string` ;
- chaque `.Result` devient un `await` : `await inbound.ReadAsync(ct)` et `await CallLlmAsync(prompt, ct)` ;
- un `CancellationToken` voyage de bout en bout — bonus de cohérence avec la série.

**Pourquoi le deadlock disparaît** : `await` ne bloque pas le thread — il **rend la main**. Le thread retourne au pool, la continuation s'exécute quand la tâche finit, sur un thread disponible. Le pipeline reste fluide sous charge, et le verdict PROPRE du Verifier confirme que l'analyseur reconnaît la correction.

Quant au *code fixer* d'IDE (l'ampoule qui transformerait `.Result` en `await` d'un clic) : il vit côté IDE, invisible de `dotnet build` — le corriger automatiquement exige de réécrire la méthode englobante en `async`, une transformation trop contextuelle pour être montrée proprement ici. Le fix manuel ci-dessus est celui que l'ampoule elle-même proposerait.

### Exercice 2 — Prouver l'exemption : un `Outcome<T>` monadique ne doit PAS être signalé

L'étage sémantique de l'analyseur (`MetadataName is "Task" or "Task`1"`) existe pour éviter les faux positifs : un type fonctionnel `Outcome<T>` qui expose une propriété `Result` est **légal et non bloquant** — l'analyseur ne doit pas le signaler.

**Objectif** : le prouver par l'exécution, puis expliquer la ligne qui fait l'exemption.

**Indices** :
- `# Indice` : la cellule ci-dessous définit un `Outcome<T>` minimal avec une propriété `Result` — elle compile et s'exécute sans avertissement ; pourquoi le build du Demo n'a-t-il rien dit sur elle ?
- `# Etape 1` : écrire un source de test `samples/exercice2.cs` utilisant `Outcome<string>.Result`, et le passer au Verifier — verdict attendu : PROPRE.
- `# Etape 2` : dans `TaskResultBlockAnalyzer.AnalyzeNode`, identifier la ligne exacte qui évite le faux positif, et formuler en une phrase ce qu'elle vérifie.
- `# Etape 3` (pour aller plus loin) : supprimer mentalement cette ligne — quels nouveaux diagnostics apparaîtraient dans le corpus de la série ?

In [8]:
// Exercice 2 -- terrain : un Outcome<T> monadique avec propriete .Result.
// Ce type N'EST PAS une Task : bloquer n'a pas de sens ici, et l'analyseur
// ne le signale pas. TODO etudiant : verifier ce verdict via le Verifier.
public readonly record struct Outcome<T>(bool IsOk, T Value)
{
    public T Result => Value;   // propriete .Result SANS Task -- legal
}

var o = new Outcome<string>(true, "aucun blocage ici");
Console.WriteLine($"Outcome<string>.Result = {o.Result}");
Console.WriteLine("Exercice a completer : passer ce type au Verifier et nommer la ligne de l'exemption");

Outcome<string>.Result = aucun blocage ici


Exercice a completer : passer ce type au Verifier et nommer la ligne de l'exemption


## C — Le contraste Python : le même défaut, deux écosystèmes

La classe de défaut est universelle : **bloquer une tâche asynchrone depuis du code synchrone**. En Python, une coroutine qui appelle une fonction bloquante (`time.sleep`, `requests.get`) produit exactement le même gel de l'événementiel. Voyons ce que chaque écosystème propose.

In [9]:
// Verifie sur cette machine : ruff est-il disponible sans installation ?
var probe = Shell.Run(here, "where", "ruff");
var ruffPresent = probe.Contains("ruff", StringComparison.OrdinalIgnoreCase);
Console.WriteLine(probe);
Console.WriteLine(ruffPresent
    ? "-> ruff present sur cette machine"
    : "-> ruff ABSENT : en Python le garde-fou est un outil a installer soi-meme");

Information : impossible de trouver des fichiers pour le(s) modèle(s) spécifié(s).



-> ruff ABSENT : en Python le garde-fou est un outil a installer soi-meme


### La comparaison, vérifiée

Sur cette machine, `ruff` n'est pas installé — ce qui est déjà la moitié de la thèse : en Python, le garde-fou est un **outil à adopter** (pip install, config, pipeline). La règle qui couvrirait notre défaut existe et est documentée : **`ASYNC101`** (famille `flake8-async`, intégrée à ruff) — « les fonctions async ne doivent pas appeler de méthodes synchrones bloquantes ». Vérifié contre la documentation ruff : cette règle fait partie des règles `ASYNC`, **non activées par défaut** — il faut les sélectionner explicitement (`[tool.ruff.lint] select = ["ASYNC"]`).

| Aspect | Python (ruff `ASYNC101` / mypy) | .NET (Roslyn `AGENTGUARD001`) |
|---|---|---|
| Où vit le garde-fou | **Hors** compilation — outil distinct | **Dans** la compilation |
| Adoption | installer + configurer + ajouter au pipeline | une `ProjectReference` |
| Activation | opt-in par règle (`select = ["ASYNC"]`) | `isEnabledByDefault: true` |
| Moment du verdict | quand on lance le lint | à chaque `dotnet build` |
| Compréhension des types | partielle (analyse statique du source) | **totale** (modèle sémantique du compilateur) |
| Faux positifs `Result<T>` | possibles (analyse sans compilateur) | évités (l'analyseur demande au compilateur) |

La nuance honnête : l'écosystème Python compense par la **richesse des règles prêtes à l'emploi** (des centaines, maintenues par la communauté) là où notre analyseur est une règle maison de 40 lignes. Le déplacement profond n'est pas la quantité mais **le moment** : en .NET, le diagnostic est *constitutif* de la chaîne qui produit le binaire — impossible de compiler sans passer devant le garde-fou.

## D1 -- AGENTGUARD005 livre : sync-over-async, deux variantes

Le code genere par agent attrape souvent la variante **`.GetAwaiter().GetResult()`** de la famille sync-over-async -- exactement le meme defaut que `.Result` (AGENTGUARD001), mais emprunte par un chemin syntaxique distinct. L'agent voit un appel de methode ordinaire ; il ne se doute pas qu'il traverse la machine a etats d'une `Task` et bloque son thread.

Deux analyseurs sont livres dans `AgentGuard.Analyzers/` :

- **`SyncOverAsyncAnalyzer`** (AGENTGUARD005) : filtre syntaxique (`GetAwaiter().GetResult()`) **plus** filtre semantique (le receiver du `GetAwaiter()` est-il `System.Threading.Tasks.Task` ou `Task<T>` ?). Les awaiters personalises et les `ValueTask` sont exemptes par construction.
- **`SyncOverAsyncConfigureAwaitAnalyzer`** (AGENTGUARD005b) : variante pedagogique dediee, capture `tache.ConfigureAwait(bool).GetAwaiter().GetResult()`. Le message diagnostique explique **pourquoi `ConfigureAwait(false)` ne sauve pas** : il reduit la capture du `SynchronizationContext`, mais `.GetAwaiter().GetResult()` bloque toujours le thread. Le sync-over-async reste entier.

In [1]:
// D1 -- les deux analyseurs, cote a cote.
// On isole les clauses qui font la difference entre les deux :
// le pivot syntaxique (GetResult), le filtre semantique (Task / Task<T>),
// et -- pour AGENTGUARD005b -- le literal bool en argument du ConfigureAwait.

var path005  = Path.Combine(here, "AgentGuard.Analyzers", "SyncOverAsyncAnalyzer.cs");
var path005b = Path.Combine(here, "AgentGuard.Analyzers", "SyncOverAsyncConfigureAwaitAnalyzer.cs");
var src005  = File.ReadAllText(path005);
var src005b = File.ReadAllText(path005b);

// 1. Filtre semantique commun aux deux analyseurs (borne anti-faux-positif).
//    Le receiver du GetAwaiter() doit etre System.Threading.Tasks.Task
//    ou Task<T> (MetadataName 'Task' ou 'Task`1') -- pas un awaiter custom.
Console.WriteLine("// --- borne semantique partagee (AGENTGUARD005 + 005b) ---");
Console.WriteLine("// MetadataName is not ("Task" or "Task`1") -> un awaiter custom n'est pas Task");

// 2. Pivot distinctif de AGENTGUARD005b : le literal bool de ConfigureAwait.
//    AGENTGUARD005 ne regarde que GetAwaiter().GetResult() ;
//    AGENTGUARD005b ajoute un troisieme filtre : ConfigureAwait(bool) doit etre
//    appele avec un LITERAL BOOL (true/false). Un argument variable ou une
//    expression ne releve pas du scope (analyse semantique excede le bug #13842).
Console.WriteLine();
Console.WriteLine("// --- pivot distinctif de AGENTGUARD005b ---");
Console.WriteLine("// Le troisieme filtre de AGENTGUARD005b exige un literal bool :");
// Console.WriteLine("//   if (cev.Arguments[0].Expression is not LiteralExpressionSyntax)");
// Console.WriteLine("//       return;  -- pas un literal -> pas signale");

// 3. Message diagnostique : la valeur du literal est passee en argument
//    au format string, pour que l'auteur du code voie explicitement que
//    ConfigureAwait(false) NE SAUVE PAS du sync-over-async.
Console.WriteLine();
Console.WriteLine("// --- message diagnostique de AGENTGUARD005b ---");
Console.WriteLine("// Format string du Rule :");
Console.WriteLine("// "ConfigureAwait({0}) ne protege pas du sync-over-async :"");

// Demonstration finale : verifier que les deux analyseurs ont bien des
// diagnostic_id distincts, comme attendu du registre shipped.
Console.WriteLine();
Console.WriteLine("// --- diagnostic_id distincts ---");
Console.WriteLine("// AGENTGUARD005 : SyncOverAsyncAnalyzer.cs");
Console.WriteLine("// AGENTGUARD005b : SyncOverAsyncConfigureAwaitAnalyzer.cs");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Repertoire de travail : Aspire


In [2]:
// D1 -- verdicts reels sur les 6 terrains SyncOverAsync committes.
// 5 terrains AGENTGUARD005 + 1 terrain AGENTGUARD005b. On observe le
// contraste entre les 3 rouges (fautifs) et les 3 propres (corriges /
// exemptes par le filtre semantique ou syntaxique).

var verdicts005 = Shell.Run(here, "dotnet",
    "run --project AgentGuard.Verifier -- " +
    "AgentGuard.Verifier/samples/SyncOverAsyncFautif.cs " +
    "AgentGuard.Verifier/samples/SyncOverAsyncCorrige.cs " +
    "AgentGuard.Verifier/samples/SyncOverAsyncGenericFautif.cs " +
    "AgentGuard.Verifier/samples/SyncOverAsyncSansGetResult.cs " +
    "AgentGuard.Verifier/samples/SyncOverAsyncAwaiterPersonnalise.cs " +
    "AgentGuard.Verifier/samples/SyncOverAsyncConfigureAwaitFautif.cs");
Console.WriteLine(verdicts005);


[SyncOverAsyncFautif.cs] VERDICT : 1 diagnostic(s) -- AGENTGUARD005 x1
    AGENTGUARD005 @ 21:9  GetAwaiter().GetResult() bloque une Task/Task de maniere synchrone : remplacer par await
[SyncOverAsyncCorrige.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche
[SyncOverAsyncGenericFautif.cs] VERDICT : 1 diagnostic(s) -- AGENTGUARD005 x1
    AGENTGUARD005 @ 21:16  GetAwaiter().GetResult() bloque une Task/Task`1 de maniere synchrone : remplacer par await
[SyncOverAsyncSansGetResult.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche
[SyncOverAsyncAwaiterPersonnalise.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche
[SyncOverAsyncConfigureAwaitFautif.cs] VERDICT : 2 diagnostic(s) -- AGENTGUARD005b x2
    AGENTGUARD005b @ 25:16  ConfigureAwait(False) ne protege pas du sync-over-async : .GetAwaiter().GetResult() bloque toujours le thread, remplacer par await
    AGENTGUARD005b @ 32:16  ConfigureAwait(True) ne protege pas du sync-over-async : .GetAwaiter().GetResu

### Lecture des verdicts

Les trois rouges sont tous du meme defaut (`GetAwaiter().GetResult()` synchrone sur une `Task`), portes par trois formes differentes : **non-generique** (`SyncOverAsyncFautif`), **generique** (`SyncOverAsyncGenericFautif` -- `Task<string>`), **avec `ConfigureAwait`** (`SyncOverAsyncConfigureAwaitFautif` -- qui declenche AGENTGUARD005b mais pas AGENTGUARD005, parce que le receiver du `GetAwaiter` est `ConfiguredTaskAwaitable`, pas `Task`).

Les trois propres relevent des trois clauses d'exemption :

- **`SyncOverAsyncCorrige`** : methode `async` + `await`. Plus de chaine fautive -- l'analyseur laisse passer.
- **`SyncOverAsyncSansGetResult`** : la chaine accede a `IsCompleted`, pas a `GetResult()`. Filtre syntaxique du membre invoque -> l'analyseur ne se declenche pas.
- **`SyncOverAsyncAwaiterPersonnalise`** : `MonAwaitable.GetAwaiter().GetResult()`. Le filtre semantique verifie que le receiver du `GetAwaiter()` est `System.Threading.Tasks.Task` ; ici c'est `MonAwaitable`. Pas de diagnostic.

**Le message d'AGENTGUARD005b est explicite** (contrairement a AGENTGUARD001) : il dit *"`ConfigureAwait(false)` ne protege pas du sync-over-async"*. C'est la valeur d'un analyseur dedie plutot qu'une regle : expliquer, pas seulement hurler.

## D — Le réflexe pour du code généré par agent

Résumé du geste complet, tel qu'un pipeline d'agent .NET peut l'adopter :

1. **Un analyseur par classe de défaut** — ici le blocage de Task ; demain l'appel HTTP sans timeout (leçon du notebook 01), le secret en dur (leçon de la série sécurité), la désérialisation non bornée. Quarante lignes chacune.
2. **Référencé dans le projet** — `OutputItemType="Analyzer"` : chaque `dotnet build`, de chaque machine, rend le verdict. L'agent qui génère du code reçoit le diagnostic **dans la sortie de son propre build** — boucle de correction immédiate, sans review humain dans la boucle.
3. **Exécutable en CI sur du code volatile** — le canal API (Verifier) permet de filtrer le code généré *avant même* qu'il n'atteigne un projet : génération → compilation en mémoire → verdict → garder ou régénérer.

C'est la réponse .NET à la question du garde-fou de code généré : ne pas ajouter un outil à côté du compilateur — **mettre la règle dans le compilateur**.

## D2 — `AGENTGUARD002` livré : `async void` hors gestionnaire, l'exemption démontrée

Second pattern d'agent classique : `async void FaitUneChose()` — la méthode ressemble à une `async Task`, mais ses exceptions **échappent à tout mécanisme d'attente** : non observables, elles font planter le process ; la tâche n'est pas attendable, donc non testable.

Ce que l'exercice 3 de la première version demandait d'écrire est désormais **livré** : `AgentGuard.Analyzers` contient l'analyseur réel (`AsyncVoidAnalyzer`, ~80 lignes), le terrain fautif vit dans le Demo (`AgentFireAndForget.SurveillerCanalAsync` — déjà signalé par le build de la section A3), et trois sources de test sont committées côté Verifier. La question pédagogique se déplace : ce qui est intéressant maintenant n'est plus « écrire le filtre » mais **l'exemption** — comment distinguer un `async void` fautif d'un gestionnaire d'événement légitime, dont le contrat C# EXIGE `void` ?

La clé : l'exemption est **sémantique**, pas lexicale. On ne regarde pas le nom du paramètre (`sender` ne prouve rien) — on demande au compilateur si la signature est `(object, T)` avec `T` qui dérive de `System.EventArgs`.


In [10]:
// D2 : l'analyseur AGENTGUARD002 en action -- trois terrains, trois verdicts.
// D'abord l'exemption, telle qu'ecrite dans AsyncVoidAnalyzer.cs :
var analyzerPath = Path.Combine(here, "AgentGuard.Analyzers", "AsyncVoidAnalyzer.cs");
var analyzerSrc = File.ReadAllText(analyzerPath);
var start = analyzerSrc.IndexOf("private static bool EstHandlerEvenement");
Console.WriteLine("// --- l'exemption, extraite de AsyncVoidAnalyzer.cs ---");
Console.WriteLine(analyzerSrc[start..analyzerSrc.IndexOf("private static bool EstEventArgsOuDerive")].TrimEnd());

// Puis les trois terrains passes au Verifier (canal API) :
var verdicts002 = Shell.Run(here, "dotnet",
    "run --project AgentGuard.Verifier -- " +
    "AgentGuard.Verifier/samples/AsyncVoidFautif.cs " +
    "AgentGuard.Verifier/samples/AsyncVoidCorrige.cs " +
    "AgentGuard.Verifier/samples/AsyncVoidHandlerExempt.cs");
Console.WriteLine();
Console.WriteLine(verdicts002);


// --- l'exemption, extraite de AsyncVoidAnalyzer.cs ---


private static bool EstHandlerEvenement(MethodDeclarationSyntax method, SyntaxNodeAnalysisContext ctx)
    {
        var parameters = method.ParameterList.Parameters;
        if (parameters.Count < 2) return false;

        // Premier parametre exactement object (le "sender").
        var senderType = ctx.SemanticModel.GetTypeInfo(parameters[0].Type).Type;
        if (senderType?.SpecialType != SpecialType.System_Object) return false;

        // Second parametre derive de System.EventArgs (ou l'est lui-meme).
        var argsType = ctx.SemanticModel.GetTypeInfo(parameters[1].Type).Type;
        if (argsType is null || argsType.TypeKind == TypeKind.Error) return false;
        return EstEventArgsOuDerive(argsType, ctx.Compilation);
    }


[AsyncVoidFautif.cs] VERDICT : 1 diagnostic(s) -- AGENTGUARD002 x1
    AGENTGUARD002 @ 13:30  La methode async void 'TraiterCommande' echappe a toute attente -- exceptions non observees, process mort
[AsyncVoidCorrige.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche
[AsyncVoidHandlerExempt.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche



### Lecture : l'exemption qui fait la différence entre un garde-fou et un gêneur

Trois verdicts, et le troisième est le plus important :

- `AsyncVoidFautif.cs` : **AGENTGUARD002 x1** — `TraiterCommande(string)` n'a pas de signature de handler, elle est signalée à sa position exacte ;
- `AsyncVoidCorrige.cs` : **PROPRE** — `async Task` au lieu d'`async void`, la méthode redevient attendable, composable, testable ;
- `AsyncVoidHandlerExempt.cs` : **PROPRE** — deux `async void` pourtant bien présents (`OnTimerElapsed`, `OnProgressReported`), et **aucun signal**. C'est l'exemption au travail : le premier reçoit `(object, EventArgs)` — la forme canonique ; le second `(object, ProgressEventArgs)` — un `EventArgs` **dérivé**, dont l'héritage est remonté par le modèle sémantique (`BaseType` en boucle, dans `EstEventArgsOuDerive`). Un grep sur `async void` ne peut pas faire cette distinction ; un lint sans compilateur doit se fier au nom des paramètres. L'analyseur demande au compilateur.

C'est le même écart de précision que l'exemption `Result<T>` de l'exercice 2, sur le second analyseur : l'étage sémantique est ce que .NET achète contre tout outillage externe.


### Exercice 3 — `AGENTGUARD003` : `Task.Run` feu, la tâche non observée

Troisième pattern d'agent : le « lance et oublie » **correct en apparence** — `Task.Run(() => ...)`. Contrairement à `async void`, la signature est honnête (une `Task` est rendue)... mais le code généré l'ignore (`_ =` absent, variable jamais attendue). La tâche s'exécute, personne n'observe ses exceptions : le `AggregateException` final frappe le finalizer.

**Objectif** : écrire le squelette du troisième analyseur.

**Indices** :
- `# Indice` : s'abonner à `SyntaxKind.InvocationExpression` ; vérifier que la méthode appelée est `Task.Run` (modèle sémantique — même filtre `MetadataName` que AGENTGUARD001).
- `# Etape 1` : distinguer les trois consommateurs d'une `Task` — `await` (observée), affectation à une variable ou `_ =` (explicite, assumée), invocation nue (feu). Seule la dernière forme est signalée.
- `# Etape 2` : écrire le squelette ci-dessous en code réel dans une copie de l'analyseur, avec son `DiagnosticDescriptor` (`AGENTGUARD003`).
- `# Etape 3` : le tester sur un source contenant `Task.Run(() => Travail())` nu, puis `_ = Task.Run(...)` — verdict attendu : rouge sur le premier, PROPRE sur le second.


In [11]:
// Exemple guide resolu -- l'analyseur AGENTGUARD003 (~70 lignes) livre dans
// `AgentGuard.Analyzers/TaskRunFireAnalyzer.cs`, sur le meme squelette a
// deux etages que ses predecesseurs : syntaxique (parent == ExpressionStatement)
// puis semantique (System.Threading.Tasks.Task.Run seulement).

var analyzerSrc = File.ReadAllText(Path.Combine(here, "AgentGuard.Analyzers", "TaskRunFireAnalyzer.cs"));
Console.WriteLine(analyzerSrc);

using System.Collections.Immutable;
using Microsoft.CodeAnalysis;
using Microsoft.CodeAnalysis.CSharp;
using Microsoft.CodeAnalysis.CSharp.Syntax;
using Microsoft.CodeAnalysis.Diagnostics;

namespace AgentGuard.Analyzers;

/// <summary>
/// AGENTGUARD003 : invocation nue de Task.Run, tache non observee.
///
/// Troisieme pattern typique du code genere par agent : ecrire
/// `Task.Run(() => Travail())` comme enonce autonome. La signature est
/// honnete (la methode rend une Task, pas void), MAIS la tache resultante
/// n'est ni attendue (await), ni affectee a une variable, ni retournee,
/// ni explicitement ignoree via discard (`_ =`). Elle s'execute en arriere-
/// plan ; ses exceptions ne sont observees par personne. A la finalisation
/// d'une telle tache fautive, le runtime declenche
/// `TaskScheduler.UnobservedTaskException`, un evenement qui porte une
/// `AggregateException` collectant les exceptions internes -- le defaut
/// (.NET 4.5+) est d'absorber l'evenement et de laisser 

## Conclusion

Ce notebook a démontré, exécution à l'appui :

- quatre **analyseurs Roslyn réels** attrapent des défauts distincts du code d'agent généré : blocage synchrone, `async void`, `Task.Run` non observé et perte de propagation d'un `CancellationToken` ;
- sur le **canal build**, `dotnet build` rend `AGENTGUARD001/002/003/004` sans outil supplémentaire — la thèse DANS-la-compilation ;
- sur le **canal API**, le `Verifier` compile les terrains fautifs, corrigés et homonymes, puis rend les mêmes diagnostics que l'IDE et MSBuild ;
- chaque analyseur combine un périmètre précis avec des exemptions exécutées, plutôt qu'une heuristique sur le nom d'une méthode ou d'un paramètre ;
- le contraste Python reste un déplacement de moment : là où ruff exige adoption et activation par règle, l'analyseur .NET est constitutif de la chaîne de build.

Les exercices restent non résolus et étendent les mêmes filtres sémantiques : `GetAwaiter().GetResult()`, l'exemption `Result<T>`, `Task.Factory.StartNew` et, désormais, la portée d'un token dans une fonction locale.

**Pour aller plus loin** : la série Aspire fournit le terrain d'agent ; l'Epic #10473 tient la parité Python/C# des garde-fous ; et la documentation Roslyn `DiagnosticAnalyzer` couvre les code fixers et les tests (`Microsoft.CodeAnalysis.Analyzer.Testing`).

See #10473

Verifions le verdict sur les cinq terrains committEs -- fautif rouge, await propre, affectation_propre (affectee a une variable ET retournee a l'appelant, aucun .Result/.Wait declenche, PROPRE pour les trois analyseurs), discard explicite propre, homonyme custom propre.

In [12]:
// Exemple guide resolu -- AGENTGUARD003 dans le Verifier.
// Cinq terrains, cinq verdicts : fautif rouge, await propre, affectation
// propre (affectee a une variable ET retournee a l'appelant, PROPRE pour
// les trois analyseurs), discard explicite propre, homonyme custom propre.
var verdicts003 = Shell.Run(here, "dotnet",
    "run --project AgentGuard.Verifier -- " +
    "AgentGuard.Verifier/samples/TaskRunFireFautif.cs " +
    "AgentGuard.Verifier/samples/TaskRunFireCorrige.cs " +
    "AgentGuard.Verifier/samples/TaskRunFireAssignation.cs " +
    "AgentGuard.Verifier/samples/TaskRunFireDiscard.cs " +
    "AgentGuard.Verifier/samples/TaskRunFireHomonyme.cs");
Console.WriteLine(verdicts003);

[TaskRunFireFautif.cs] VERDICT : 1 diagnostic(s) -- AGENTGUARD003 x1
    AGENTGUARD003 @ 14:9  Task.Run 'Task.Run(() => Console.WriteLine("ping"))' lance une tache non observee -- exceptions perdues, comportement indefini
[TaskRunFireCorrige.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche
[TaskRunFireAssignation.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche
[TaskRunFireDiscard.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche
[TaskRunFireHomonyme.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche



### Lecture des verdicts

Cinq terrains, cinq verdicts. Le seul rouge attendu sur AGENTGUARD003 est le fautif nu -- exactement ce que rend le Verifier :

- `TaskRunFireFautif.cs` : **AGENTGUARD003 x1** sur la ligne de `Task.Run(...)` en enonce autonome. La position est precise (colonne 9, la ou commence l'invocation), le message cite l'expression entiere ;
- `TaskRunFireCorrige.cs` : **PROPRE** -- `await Task.Run(...)` place l'invocation comme fils d'un `AwaitExpressionSyntax`, donc le parent n'est plus un `ExpressionStatement` nu. L'exemption tombe naturellement ;
- `TaskRunFireAssignation.cs` : **PROPRE** (tache affectee a une variable ET retournee a l'appelant -- la double recuperation (assignation + return) garantit la responsabilite transmise, aucun .Result/.Wait declenche) ;
- `TaskRunFireDiscard.cs` : **PROPRE** -- `_ = Task.Run(...)` place l'invocation comme `Right` d'un `AssignmentExpressionSyntax` avec `Left = IdentifierNameSyntax("_")`. L'exemption couvre cette forme volontairement assumee ;
- `TaskRunFireHomonyme.cs` : **PROPRE** -- `MonRunner.Run` n'a pas `System.Threading.Tasks.Task` comme `ContainingType`, donc le filtre semantique le tranche des l'etage 1.

C'est le **meme** analyseur qui rend le verdict dans le canal build (l'avertissement `AGENTGUARD003` a la section A3 sur `Program.cs(75,9)`) et dans le canal API : un moteur, deux canaux -- la these de la section A3, demontree une troisieme fois.


### Exercice 4 -- Etendre AGENTGUARD003 : signaler aussi `Task.Factory.StartNew(...)` nu

Le filtre semantique actuel ne vise **que** `Task.Run` -- c'est delibere : la surcharge `Task.Factory.StartNew(Action)` est l'API historique equivalente (avant C# 4.0, c'etait la seule facon de planifier un delegue). Les agents la produisent parfois quand leur fenetre d'entrainement est large, ou quand l'utilisateur importe `using System.Threading.Tasks;` sans s'en rendre compte et laisse l'IDE proposer la forme la plus familiere.

**Objectif** : etendre `TaskRunFireAnalyzer.AnalyzeInvocation` pour signaler aussi l'invocation nue `Task.Factory.StartNew(...)`, sans casser les exemptions existantes (await, affectation, discard, return, homonyme).

**Indices** :
- `# Indice` : l'invocation a surveiller est de la forme `Task.Factory.StartNew(...)`. Le recepteur de `StartNew` est un `MemberAccessExpression` (`Task.Factory`), pas une simple reference a `Task`. La detection semantique change : il faut que le symbole invoque soit `StartNew` ET que son `ContainingType` soit `TaskFactory` (la propriete statique `Task.Factory` rend un `TaskFactory` ; sa methode `StartNew` est definie sur `TaskFactory`, pas sur `Task`).
- `# Indice` : pour ne pas casser la regle actuelle, ecrire le filtre en **ajout** : signaler `Task.Run(...)` OU `Task.Factory.StartNew(...)`. Tester en re-compilant un sample `TaskFactoryStartNewNude.cs` qui contient `Task.Factory.StartNew(() => Travail())` nu -- verdict attendu : AGENTGUARD003.
- `# Etape 1` : creer `samples/TaskFactoryStartNewNude.cs` avec un enonce `Task.Factory.StartNew(() => Console.WriteLine("ping"));` et verifier au Verifier qu'il rend PROPRE aujourd'hui (Task.Factory.StartNew est hors scope).
- `# Etape 2` : dupliquer le filtre semantique dans `TaskRunFireAnalyzer.cs` (ou creer un second analyseur `TaskFactoryStartNewAnalyzer.cs` si l'on veut garder la separation). Le verdict attendu apres extension : AGENTGUARD003 x1 sur la ligne fautive.
- `# Etape 3` : confirmer que les cinq terrains existants continuent a rendre les memes verdicts -- l'extension ne doit pas faireRegression sur AGENTGUARD001 ni AGENTGUARD002.


In [13]:
// Exercice 4 a completer -- etendre AGENTGUARD003 a Task.Factory.StartNew.
// Indice : le filtre semantique actuel vise Task.Run ; pour Task.Factory,
// StartNew est defini sur TaskFactory (pas Task), mais la propriete
// statique Task.Factory est elle-membre de Task -- d'ou le double saut.
// TODO etudiant : ajouter la detection et tester sur un nouveau sample.
Console.WriteLine("Exercice a completer : etendre AGENTGUARD003 a Task.Factory.StartNew nu");

Exercice a completer : etendre AGENTGUARD003 a Task.Factory.StartNew nu


## D3 — `AGENTGUARD004` : l'annulation doit traverser toute la chaîne

Un agent reçoit souvent un `CancellationToken` depuis une requête HTTP ou un `BackgroundService`. Le token n'est utile que s'il voyage jusqu'à chaque opération annulable. Le défaut est discret : la méthode englobante possède bien le token, la cible l'accepte, mais l'appel omet l'argument optionnel. Le code compile et continue à travailler après l'annulation demandée.

`AGENTGUARD004` vise exactement cette rupture. Il ne cherche ni un nom comme `Async`, ni un paramètre nommé `ct` : il demande au modèle sémantique si le symbole englobant possède un vrai `System.Threading.CancellationToken`, puis si la signature cible expose ce même type et si l'argument a été fourni explicitement.

In [14]:
// Exemple guide resolu -- lire l'analyseur semantique livre.
var analyzer004 = File.ReadAllText(Path.Combine(
    here, "AgentGuard.Analyzers", "CancellationTokenPropagationAnalyzer.cs"));
Console.WriteLine(analyzer004);

using System.Collections.Immutable;
using System.Linq;
using Microsoft.CodeAnalysis;
using Microsoft.CodeAnalysis.Diagnostics;
using Microsoft.CodeAnalysis.Operations;

namespace AgentGuard.Analyzers;

/// <summary>
/// AGENTGUARD004 : perte d'un CancellationToken disponible.
///
/// Une méthode d'agent reçoit souvent un token depuis la requête HTTP ou le
/// BackgroundService. Si elle appelle une opération dont la signature expose
/// explicitement CancellationToken mais omet cet argument optionnel, l'arrêt
/// demandé ne traverse plus la chaîne. L'analyse est entièrement sémantique :
/// le type doit être System.Threading.CancellationToken, sans heuristique sur
/// le nom de la méthode ni du paramètre.
/// </summary>
[DiagnosticAnalyzer(LanguageNames.CSharp)]
public sealed class CancellationTokenPropagationAnalyzer : DiagnosticAnalyzer
{
    public const string DiagnosticId = "AGENTGUARD004";

    private static readonly DiagnosticDescriptor Rule = new(
        DiagnosticId,
        

### Mesure — un fautif, quatre contrôles négatifs

Le Verifier soumet cinq sources au même analyseur :

1. token disponible + cible annulable + argument omis : **rouge** ;
2. token transmis : propre ;
3. aucun token disponible : propre ;
4. surcharge réellement choisie sans paramètre token : propre ;
5. type homonyme `AgentCustom.CancellationToken` : propre.

Ce dernier contrôle est décisif : le filtre compare les symboles au type BCL `System.Threading.CancellationToken`, pas leur simple orthographe.

In [15]:
var verdicts004 = Shell.Run(here, "dotnet",
    "run --project AgentGuard.Verifier -- " +
    "AgentGuard.Verifier/samples/CancellationTokenFautif.cs " +
    "AgentGuard.Verifier/samples/CancellationTokenCorrige.cs " +
    "AgentGuard.Verifier/samples/CancellationTokenSansTokenDisponible.cs " +
    "AgentGuard.Verifier/samples/CancellationTokenSurchargeSansToken.cs " +
    "AgentGuard.Verifier/samples/CancellationTokenHomonyme.cs");
Console.WriteLine(verdicts004);

[CancellationTokenFautif.cs] VERDICT : 1 diagnostic(s) -- AGENTGUARD004 x1
    AGENTGUARD004 @ 10:15  L'appel à 'Task AgentCancellationFautif.EnvoyerAuModeleAsync(string prompt, CancellationToken cancellationToken = default(CancellationToken))' omet CancellationToken alors que 'cancellationToken' est disponible dans la méthode englobante
[CancellationTokenCorrige.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche
[CancellationTokenSansTokenDisponible.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche
[CancellationTokenSurchargeSansToken.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche
[CancellationTokenHomonyme.cs] VERDICT : PROPRE -- aucun garde-fou AgentGuard declenche



### Lecture du résultat

Le terrain fautif rend exactement **`AGENTGUARD004 x1`** sur l'appel qui perd l'annulation. Les quatre autres terrains rendent **PROPRE** : le diagnostic disparaît dès que le vrai token est transmis, et il n'apparaît pas quand le contrat ou la portée ne permettent aucune propagation.

La distinction « surcharge sans token » évite une recommandation impossible : si la méthode réellement résolue par Roslyn n'expose pas `CancellationToken`, l'analyseur ne prétend pas qu'un argument pourrait lui être ajouté. La distinction « homonyme » évite le faux positif lexical. Ces résultats mesurés bornent précisément ce que garantit la règle : **propager un token déjà disponible vers une cible qui l'accepte explicitement**.

### Exercice 5 — Borner la portée dans une fonction locale

L'analyseur remonte les symboles englobants : une fonction locale ou une lambda peut capturer le token de sa méthode parente. Cette propriété évite de perdre l'annulation dans un callback interne, mais elle doit rester précise.

**Objectif** : écrire un terrain qui prouve la capture, puis un contre-exemple où un token local est déjà transmis.

- `# Etape 1` : dans une méthode `TraiterAsync(CancellationToken ct)`, déclarer une fonction locale `EtapeAsync()` qui appelle une cible annulable sans `ct` ; verdict attendu : `AGENTGUARD004`.
- `# Etape 2` : transmettre `ct` depuis la fonction locale ; verdict attendu : PROPRE.
- `# Indice` : ne modifiez pas les noms des méthodes pour aider l'analyseur — il ne les utilise pas.
- `# TODO etudiant` : ajouter les deux sources aux samples du Verifier et expliquer quel `ContainingSymbol` rend la capture visible.

In [16]:
// Exercice 5 a completer -- terrain minimal de fonction locale.
// TODO etudiant : transformer ce commentaire en deux samples du Verifier.
//
// static async Task TraiterAsync(CancellationToken ct)
// {
//     async Task EtapeAsync()
//     {
//         await OperationAnnulableAsync();      // attendu : AGENTGUARD004
//         // puis corriger avec OperationAnnulableAsync(ct)
//     }
//     await EtapeAsync();
// }
Console.WriteLine("Exercice a completer : verifier la propagation dans une fonction locale");

Exercice a completer : verifier la propagation dans une fonction locale
